# 04. 문서 이미지 → 질문답변(Document QA)

**실습 목표**  
스캔 문서나 영수증 이미지에서 질문에 해당하는 텍스트를 찾아 답한다.

**주요 Hugging Face 모델**  
`impira/layoutlm-document-qa`

> 이 노트북은 **파인튜닝 없이 사전학습 모델을 추론에 활용**하는 실습이다.  
> RTX 4060 8GB 환경을 고려했으며, CUDA가 없으면 CPU로 자동 전환하도록 구성하였다.

In [3]:
# uv add transformers accelerate pillow requests pytesseract

In [4]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else -1
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", TORCH_DEVICE)

PyTorch: 2.14.0+cu126
CUDA available: True
device: cuda


In [5]:
from PIL import Image
import requests
from io import BytesIO

def load_image(url):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")

In [6]:
DOC_URL = "https://huggingface.co/datasets/Narsil/document-question-answering-sample/resolve/main/document.png"
document = load_image(DOC_URL)
display(document)

HTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/datasets/Narsil/document-question-answering-sample/resolve/main/document.png

In [ ]:
from transformers import pipeline
doc_qa = pipeline(
    "document-question-answering",
    model="impira/layoutlm-document-qa",
    device=DEVICE
)
questions = ["What is the invoice number?", "What is the total?"]
for q in questions:
    try:
        print(q, "->", doc_qa(image=document, question=q))
    except Exception as e:
        print("환경에 따라 OCR 엔진이 추가로 필요할 수 있다:", e)

## 주의
LayoutLM 계열 파이프라인은 OCR 의존성이 있을 수 있다. Windows 환경에서 OCR 관련 오류가 발생하면 Tesseract 설치 여부를 확인해야 한다.